In [ ]:
# Create anonymized rMove datapoints file
from src.map_terminology import map_rmove_to_generic_mode
import pandas as pd

from src.calculate_speed import calculate_speed_for_dataframe

root_path = 'data/rMove/'
output_file_path = f'{root_path}rmove_processed.csv'

locations_df = pd.read_csv(f'{root_path}Location_2023.csv')
trips_df = pd.read_csv(f'{root_path}Household_Travel_Survey_Trips_-7221806773183684102.csv', low_memory=False)
trips_df = trips_df.set_index('trip_id')

dataframes = []
count = 0
for trip_id, df in locations_df.groupby('tripid'):
    if trip_id not in trips_df.index:
        continue

    df = df.drop(columns=['speed'])
    df = df.rename(columns={'lon': 'lng', 'collect_time': 'timestamp'})
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values('timestamp')
    df_with_speeds, _, _ = calculate_speed_for_dataframe(df, with_smoothing=False)

    df_with_speeds = df_with_speeds.iloc[1:] # drop first row
    df_with_speeds = df_with_speeds.rename(columns={'tripid': 'traj_id',
                                                    'distance_km': 'distance',
                                                    'speed_kmh': 'speed',
                                                    'time_diff_hr': 'time_diff'})

    # df_with_speeds['traj_id'] = f'rMove_{df_with_speeds["traj_id"]}'
    labeled_mode = trips_df.loc[trip_id]['mode_1']

    labeled_mode = map_rmove_to_generic_mode(labeled_mode)
    if labeled_mode == "other":
        continue

    df_with_speeds['label'] = labeled_mode

    df_with_speeds = df_with_speeds[['time_diff', 'speed', 'distance', 'traj_id', 'label']]

    dataframes.append(df_with_speeds)
    count += 1
    print(f"Appending row for {trip_id} ({count} files processed)")

dataframe = pd.concat(dataframes)
dataframe = dataframe.set_index('traj_id')
dataframe.to_csv(output_file_path, index=True, header=True, mode='w')

print("Done!")
print(f"Trips processed: {count}")

In [7]:
# Anonymize Spectus Database Metrics
import pandas as pd

df_spectus = pd.read_csv(f'./data/Spectus/Lyra_Processed/20260725_downsampling_masks.csv', low_memory=False)

df_spectus['trip_id'] = (df_spectus['trip_id'] != df_spectus['trip_id'].shift()).cumsum() - 1

df_spectus = df_spectus.set_index('trip_id')

df_spectus.to_csv(f'./data/Spectus/spectus_downsampling_masks_anonymized.csv', index=True, header=True, mode='w')


In [3]:
# Anonymize GeoLife Database Metrics
import pandas as pd

df_geolife = pd.read_csv(f'./data/GeoLife/20260714_geolife_database_metrics.csv')

df_geolife['trip_id'] = (df_geolife['trip_id'] != df_geolife['trip_id'].shift()).cumsum() - 1

df_geolife = df_geolife.set_index('trip_id')

df_geolife.to_csv(f'./data/GeoLife/20260714_geolife_database_metrics_anonymized.csv', index=True, header=True, mode='w')

In [6]:
# Anonymize rMove Database Metrics
import pandas as pd

df_rmove = pd.read_csv(f'./data/rMove/rmove_processed.csv', low_memory=False)

df_rmove['traj_id'] = (df_rmove['traj_id'] != df_rmove['traj_id'].shift()).cumsum() - 1

df_rmove = df_rmove.set_index('traj_id')

df_rmove.to_csv(f'./data/rMove/rmove_processed_anonymized.csv', index=True, header=True, mode='w')